# Thesis Evaluation — Multi-Model Agentic Fashion System

This notebook produces the quantitative results for comparing **three orchestration modes**:

| Mode | Orchestrator | Synthesizer |
|------|-------------|-------------|
| A    | Gemini 2.0 Flash | Gemini 2.0 Flash |
| B    | Gemini 2.0 Flash (orchestrator) | GPT-4o (synthesizer) |
| C    | GPT-4o (orchestrator) | Claude 3.5 Sonnet (synthesizer) |

**Evaluation axes:**
1. Token cost & efficiency
2. Behavioural accuracy (SR, SCR, QRR, GAS)
3. Cost-Efficiency Score (CES)
4. Statistical significance tests

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from psycopg2.extras import RealDictCursor
from scipy import stats

# Style
sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

# Database connection
conn = psycopg2.connect(
    host=os.getenv('PGHOST', 'localhost'),
    port=int(os.getenv('PGPORT', '5432')),
    dbname=os.getenv('PGDATABASE', 'fashion_rag'),
    user=os.getenv('PGUSER', 'fashion_user'),
    password=os.getenv('PGPASSWORD', ''),
    connect_timeout=5,
)
print('Connected to PostgreSQL')

## 1 · Token Cost Analysis

In [ ]:
# ── Pricing constants (USD per 1M tokens) ──────────────────────────
PRICING = {
    'gemini': {'input': 0.075, 'output': 0.30},
    'gpt':    {'input': 2.50,  'output': 10.00},
    'claude': {'input': 3.00,  'output': 15.00},
}

# Pull mode_cost_summary view
df_cost = pd.read_sql('SELECT * FROM mode_cost_summary ORDER BY orchestration_mode;', conn)
display(df_cost)

In [ ]:
# ── Per-turn cost bar chart ─────────────────────────────────────────
if not df_cost.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Average tokens per turn
    ax1 = axes[0]
    ax1.bar(df_cost['orchestration_mode'], df_cost['avg_total_tokens'], color=sns.color_palette('pastel'))
    ax1.set_title('Avg Total Tokens per Turn')
    ax1.set_xlabel('Orchestration Mode')
    ax1.set_ylabel('Tokens')

    # Average USD per turn
    ax2 = axes[1]
    ax2.bar(df_cost['orchestration_mode'], df_cost['avg_usd_per_turn'].astype(float), color=sns.color_palette('muted'))
    ax2.set_title('Avg USD Cost per Turn')
    ax2.set_xlabel('Orchestration Mode')
    ax2.set_ylabel('USD')
    ax2.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

    plt.tight_layout()
    plt.savefig('analysis/token_cost_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print('No cost data available yet — run sessions first.')

## 2 · Behavioural Accuracy Metrics

In [ ]:
# ── Pull per-session behaviour data ─────────────────────────────────
q_behaviour = """
WITH session_modes AS (
    SELECT
        s.session_id,
        s.preferred_model,
        COALESCE(st.orchestration_mode, 'unknown') AS orchestration_mode,
        s.gender,
        s.gender_hint_enabled
    FROM user_sessions s
    LEFT JOIN session_token_summary st USING (session_id)
)
SELECT
    sm.orchestration_mode,
    sm.session_id,
    sm.gender,
    sm.gender_hint_enabled,
    (SELECT COUNT(*) FROM product_impressions pi WHERE pi.session_id = sm.session_id) AS impressions,
    (SELECT COUNT(*) FROM product_clicks pc WHERE pc.session_id = sm.session_id) AS clicks,
    (SELECT COUNT(*) FROM selected_items si WHERE si.session_id = sm.session_id) AS selections,
    (SELECT COUNT(*) FROM product_intents pit WHERE pit.session_id = sm.session_id AND pit.intent_type = 'will_buy') AS will_buy,
    (SELECT COUNT(*) FROM product_intents pit2 WHERE pit2.session_id = sm.session_id AND pit2.intent_type = 'not_for_me') AS not_for_me,
    (SELECT COUNT(*) FROM user_orders uo WHERE uo.session_id = sm.session_id) AS orders,
    (SELECT COUNT(DISTINCT search_query) FROM product_impressions pi3
        WHERE pi3.session_id = sm.session_id AND pi3.search_query <> '') AS distinct_queries
FROM session_modes sm;
"""
df_sessions = pd.read_sql(q_behaviour, conn)
print(f'Total sessions: {len(df_sessions)}')
display(df_sessions.head())

In [ ]:
# ── Compute per-session derived metrics ─────────────────────────────
df = df_sessions.copy()
df['sr']  = df['selections'] / df['impressions'].replace(0, np.nan)
df['scr'] = df['will_buy']   / df['selections'].replace(0, np.nan)
df['qrr'] = (df['distinct_queries'] >= 2).astype(int)
df['converted'] = (df['orders'] > 0).astype(int)

# ── Aggregate by orchestration mode ─────────────────────────────────
agg = df.groupby('orchestration_mode').agg(
    n_sessions=('session_id', 'count'),
    mean_sr=('sr', 'mean'),
    mean_scr=('scr', 'mean'),
    mean_qrr=('qrr', 'mean'),
    conversion_rate=('converted', 'mean'),
).round(4)
display(agg)

In [ ]:
# ── Radar chart of accuracy metrics ─────────────────────────────────
if not agg.empty and len(agg) > 1:
    categories = ['SR', 'SCR', 'QRR', 'Conversion']
    modes = agg.index.tolist()

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]

    colors = sns.color_palette('Set2', len(modes))
    for i, mode in enumerate(modes):
        values = [
            agg.loc[mode, 'mean_sr'],
            agg.loc[mode, 'mean_scr'],
            agg.loc[mode, 'mean_qrr'],
            agg.loc[mode, 'conversion_rate'],
        ]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=mode, color=colors[i])
        ax.fill(angles, values, alpha=0.15, color=colors[i])

    ax.set_thetagrids([a * 180 / np.pi for a in angles[:-1]], categories)
    ax.set_title('Behavioural Accuracy by Orchestration Mode', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig('analysis/accuracy_radar.png', bbox_inches='tight')
    plt.show()
else:
    print('Need >= 2 modes with data for radar chart.')

## 3 · Gender-Appropriate Selection (GAS)

In [ ]:
# ── Category gender inference ───────────────────────────────────────
MALE_CATS   = {'Longsleeve', 'T-Shirt', 'Shirt', 'Hoodie', 'Shorts', 'Pants', 'Blazer', 'Polo'}
FEMALE_CATS = {'Dress', 'Skirt', 'Blouse', 'Top'}

q_gas = """
WITH session_modes AS (
    SELECT s.session_id, s.gender, s.gender_hint_enabled,
           COALESCE(st.orchestration_mode, 'unknown') AS orchestration_mode
    FROM user_sessions s
    LEFT JOIN session_token_summary st USING (session_id)
    WHERE s.gender IS NOT NULL
)
SELECT sm.session_id, sm.gender, sm.gender_hint_enabled,
       sm.orchestration_mode, si.label
FROM session_modes sm
JOIN selected_items si USING (session_id);
"""
df_gas_raw = pd.read_sql(q_gas, conn)

def classify_match(row):
    if row['gender'] == 'male' and row['label'] in MALE_CATS:
        return 'match'
    elif row['gender'] == 'male' and row['label'] in FEMALE_CATS:
        return 'mismatch'
    elif row['gender'] == 'female' and row['label'] in FEMALE_CATS:
        return 'match'
    elif row['gender'] == 'female' and row['label'] in MALE_CATS:
        return 'mismatch'
    return 'unisex'

if not df_gas_raw.empty:
    df_gas_raw['gender_match'] = df_gas_raw.apply(classify_match, axis=1)
    df_gendered = df_gas_raw[df_gas_raw['gender_match'] != 'unisex']

    gas_table = df_gendered.groupby(['gender_hint_enabled', 'orchestration_mode']).agg(
        gendered_selections=('gender_match', 'count'),
        matched=('gender_match', lambda x: (x == 'match').sum()),
    )
    gas_table['gas'] = (gas_table['matched'] / gas_table['gendered_selections']).round(4)
    display(gas_table)
else:
    print('No gender-annotated sessions yet.')

## 4 · Cost-Efficiency Score (CES)

In [ ]:
# CES = (α · Norm_SR + β · Norm_SCR + γ · Norm_CR) / Norm_Cost
# Default weights: α=0.4, β=0.3, γ=0.3
ALPHA, BETA, GAMMA = 0.4, 0.3, 0.3

if not agg.empty:
    ces = agg.copy()

    # Min-max normalise (0–1)
    for col in ['mean_sr', 'mean_scr', 'conversion_rate']:
        cmin, cmax = ces[col].min(), ces[col].max()
        if cmax > cmin:
            ces[f'norm_{col}'] = (ces[col] - cmin) / (cmax - cmin)
        else:
            ces[f'norm_{col}'] = 1.0

    # Merge cost data
    if not df_cost.empty:
        cost_map = df_cost.set_index('orchestration_mode')['avg_usd_per_turn'].astype(float)
        ces['avg_cost'] = ces.index.map(lambda m: cost_map.get(m, np.nan))
        cmin, cmax = ces['avg_cost'].min(), ces['avg_cost'].max()
        if cmax > cmin:
            ces['norm_cost'] = (ces['avg_cost'] - cmin) / (cmax - cmin)
        else:
            ces['norm_cost'] = 1.0

        ces['CES'] = (
            ALPHA * ces['norm_mean_sr']
            + BETA * ces['norm_mean_scr']
            + GAMMA * ces['norm_conversion_rate']
        ) / ces['norm_cost'].replace(0, 0.001)  # avoid div-by-zero

        display(ces[['n_sessions', 'mean_sr', 'mean_scr', 'conversion_rate', 'avg_cost', 'CES']].round(4))
    else:
        print('No cost data to compute CES.')
else:
    print('No accuracy data to compute CES.')

## 5 · Statistical Significance Tests

In [ ]:
# ── 5a. Chi-square test on conversion rates ────────────────────────
if not df.empty and df['orchestration_mode'].nunique() >= 2:
    contingency = pd.crosstab(df['orchestration_mode'], df['converted'])
    if contingency.shape[1] == 2:
        chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
        print(f'Chi-square test (conversion ~ mode):')
        print(f'  χ² = {chi2:.4f}, p = {p_chi:.6f}, dof = {dof}')
        if p_chi < 0.05:
            print('  → Statistically significant at α=0.05')
        else:
            print('  → NOT statistically significant at α=0.05')
    else:
        print('Insufficient variation for chi-square (all sessions converted or none).')
else:
    print('Need >= 2 modes for chi-square test.')

In [ ]:
# ── 5b. Kruskal-Wallis test on SR ──────────────────────────────────
if not df.empty and df['orchestration_mode'].nunique() >= 2:
    groups = [g['sr'].dropna().values for _, g in df.groupby('orchestration_mode')]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        h_stat, p_kw = stats.kruskal(*groups)
        print(f'Kruskal-Wallis test (SR ~ mode):')
        print(f'  H = {h_stat:.4f}, p = {p_kw:.6f}')
    else:
        print('Not enough non-empty groups for Kruskal-Wallis.')
else:
    print('Need >= 2 modes for KW test.')

In [ ]:
# ── 5c. Bootstrap 95% CI for mean SR per mode ──────────────────────
def bootstrap_ci(data, n_boot=10000, ci=0.95):
    """Return (lower, upper) bootstrap CI for the mean."""
    if len(data) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(42)
    means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.quantile(means, alpha), np.quantile(means, 1 - alpha)

if not df.empty:
    print('Bootstrap 95% CI for mean SR:')
    for mode, g in df.groupby('orchestration_mode'):
        vals = g['sr'].dropna().values
        lo, hi = bootstrap_ci(vals)
        print(f'  {mode}: [{lo:.4f}, {hi:.4f}]  (n={len(vals)})')

In [ ]:
# ── 5d. Cliff's delta (effect size) for Mode A vs others ───────────
def cliffs_delta(x, y):
    """Compute Cliff's delta between x and y."""
    x, y = np.asarray(x), np.asarray(y)
    n = len(x) * len(y)
    if n == 0:
        return np.nan
    more = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return (more - less) / n

modes = df['orchestration_mode'].unique()
if len(modes) >= 2:
    baseline = modes[0]
    base_vals = df.loc[df['orchestration_mode'] == baseline, 'sr'].dropna().values
    print(f"Cliff's delta (SR) — baseline: {baseline}")
    for m in modes[1:]:
        other_vals = df.loc[df['orchestration_mode'] == m, 'sr'].dropna().values
        d = cliffs_delta(base_vals, other_vals)
        magnitude = 'negligible'
        if abs(d) >= 0.474:
            magnitude = 'large'
        elif abs(d) >= 0.33:
            magnitude = 'medium'
        elif abs(d) >= 0.147:
            magnitude = 'small'
        print(f'  vs {m}: δ = {d:.4f} ({magnitude})')
else:
    print('Need >= 2 modes.')

## 6 · Summary Table for Thesis

In [ ]:
# ── Combined results summary ────────────────────────────────────────
if not agg.empty:
    summary = agg[['n_sessions', 'mean_sr', 'mean_scr', 'mean_qrr', 'conversion_rate']].copy()
    summary.columns = ['Sessions', 'SR', 'SCR', 'QRR', 'CR']
    
    if not df_cost.empty:
        cost_map = df_cost.set_index('orchestration_mode')[['avg_total_tokens', 'avg_usd_per_turn']]
        cost_map.columns = ['Avg Tokens/Turn', 'Avg USD/Turn']
        cost_map['Avg USD/Turn'] = cost_map['Avg USD/Turn'].astype(float)
        summary = summary.join(cost_map, how='left')
    
    if 'CES' in ces.columns:
        summary['CES'] = ces['CES'].round(4)
    
    print('\n════════════════ THESIS SUMMARY TABLE ════════════════')
    display(summary.round(4))
    
    # Export to CSV for LaTeX inclusion
    summary.to_csv('analysis/thesis_summary_table.csv')
    print('\nExported to analysis/thesis_summary_table.csv')
else:
    print('No data — run evaluation sessions first.')

In [ ]:
# ── Cleanup ─────────────────────────────────────────────────────────
conn.close()
print('Done.')